# 🦜 VIENEU-TTS v3 TURBO — STUDIO NHÂN BẢN GIỌNG NÓI TIẾNG VIỆT 48kHz
### 🔒 CLONE GIỌNG ZERO-SHOT SIÊU TỰ NHIÊN (3–8 GIÂY) & XUẤT FILE .JSON VỀ TOOL REVIEW VIDEO
*Mã nguồn gốc: [pnnbao97/VieNeu-TTS](https://github.com/pnnbao97/VieNeu-TTS) của tác giả Phạm Nguyễn Ngọc Bảo*

---

### ⚡ TÍNH NĂNG NỔI BẬT:
1. **Tốc độ GPU T4 cực nhanh**: RTF ≈ 0.02 (nhanh gấp ~50 lần thời gian thực), xử lý và nghe thử chỉ trong 1-2 giây.
2. **Chất lượng âm thanh 48 kHz**: Chuẩn Studio phát thanh, hỗ trợ tag cảm xúc `[cười]`, `[thở dài]`, `[hắng giọng]`.
3. **Khử tạp âm tự động**: Bộ lọc `denoiser.onnx` làm sạch nền clip mẫu trước khi trích xuất đặc trưng.
4. **Xuất file `.json` siêu nhẹ (~4 KB)**: Chỉ cần tải file cấu hình giọng về máy, bỏ vào thư mục `vieneu_custom_voices/` của Tool là chạy offline 100% bằng CPU!

---

### 🚀 HƯỚNG DẪN 1-CLICK:
- Bấm menu **Thời gian chạy (Runtime)** -> Chọn **Chạy tất cả (Run all)** (phím tắt `Ctrl + F9`).
- Kéo xuống ô `[BƯỚC 2/2]` để mở giao diện WebUI thao tác trực quan.


In [ ]:
# @title ⚙️ [BƯỚC 1/2] KIỂM TRA GPU T4 & CÀI ĐẶT VIENEU-TTS { display-mode: "form" }
# @markdown Bấm nút Play (hoặc nhấn phím tắt **Ctrl + F9**) để tự động cài đặt môi trường:
LUU_VAO_GOOGLE_DRIVE = True #@param {type:"boolean"}

import os, sys, time, shutil, json, warnings
warnings.filterwarnings("ignore")

print("="*65)
print("🚀 [1/2] ĐANG THIẾT LẬP MÔI TRƯỜNG VIENEU-TTS...")
print("="*65)

IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
DRIVE_DIR = "/content/drive/MyDrive/VieNeu_Studio"
LOCAL_EXPORT_DIR = "/content/vieneu_cloned_voices"

if IS_COLAB and LUU_VAO_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        print("📁 Đang kết nối Google Drive (Lưu vĩnh viễn các file giọng .json đã clone)...")
        drive.mount('/content/drive')
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print(f"✅ Đã kết nối Google Drive: {DRIVE_DIR}")
    except Exception as e:
        print(f"⚠️ Không dùng được Google Drive ({e}). Sẽ lưu vào thư mục tạm Colab.")

os.makedirs(LOCAL_EXPORT_DIR, exist_ok=True)

try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"✅ GPU Sẵn Sàng: {gpu_name} ({vram:.1f} GB VRAM) — Tốc độ cực nhanh!")
    else:
        print("⚠️ CẢNH BÁO: Đang dùng CPU. Hãy vào menu Thời gian chạy (Runtime) -> Đổi loại phần cứng -> Chọn T4 GPU!")
except Exception:
    pass

print("\n⏳ Đang cài đặt thư viện vieneu, transformers, gradio...")
!pip install -q "vieneu[cuda]" transformers soundfile gradio accelerate pyngrok
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared 2>/dev/null || true

print("\n✅ [BƯỚC 1/2] Cài đặt thành công! Tiếp tục chạy ô [BƯỚC 2/2] bên dưới.")


In [ ]:
# @title 🎙️ [BƯỚC 2/2] KHỞI CHẠY WEB UI CLONE GIỌNG NÓI & XUẤT FILE .JSON { display-mode: "form" }
# @markdown Bấm nút Play để mở giao diện WebUI trực quan:

import os, sys, time, json, re, shutil, tempfile
import numpy as np
import soundfile as sf
import gradio as gr
from vieneu import Vieneu

print("⏳ Đang nạp mô hình VieNeu-TTS v3 Turbo 48kHz vào GPU...")
tts = Vieneu()
sample_rate = getattr(tts, "sample_rate", 48000)
print(f"✅ VieNeu-TTS v3 Turbo sẵn sàng! (Backend={getattr(tts, 'backend', 'unknown')}, SR={sample_rate}Hz)")

DRIVE_DIR = "/content/drive/MyDrive/VieNeu_Studio"
LOCAL_EXPORT_DIR = "/content/vieneu_cloned_voices"
os.makedirs(LOCAL_EXPORT_DIR, exist_ok=True)
if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(DRIVE_DIR, exist_ok=True)

SESSION_CLONED_VOICES = {}

def clone_and_preview(audio_file, voice_name, gender, region, test_text, denoise):
    if not audio_file:
        return None, "⚠️ Vui lòng tải lên file âm thanh mẫu (3–8 giây)!", None
    if not voice_name or not voice_name.strip():
        voice_name = "Custom_Voice_" + str(int(time.time()))
    voice_name = voice_name.strip()
    
    if not test_text or not test_text.strip():
        test_text = "[cười] Chào bạn, đây là giọng đọc vừa được nhân bản bằng VieNeu-TTS. Âm thanh chuẩn 48kHz nghe rất tự nhiên!"
        
    try:
        tts.add_voice(
            name=voice_name,
            ref_audio=audio_file,
            denoise=bool(denoise),
            gender="male" if "Nam" in gender else ("female" if "Nữ" in gender else ""),
            description=f"Giọng {gender} · Miền {region} · Clone VieNeu"
        )
        
        voice_info = tts._preset_voices.get(voice_name)
        if not voice_info:
            return None, "⚠️ Lỗi: Không trích xuất được đặc trưng giọng!", None
            
        emb = voice_info.get("speaker_emb")
        codes = voice_info.get("codes")
        
        json_data = {
            "name": voice_name,
            "gender": "male" if "Nam" in gender else ("female" if "Nữ" in gender else "unknown"),
            "region": region,
            "description": f"Giọng {gender} · {region} (Clone từ VieNeu Colab)",
            "speaker_emb": [round(float(x), 6) for x in np.asarray(emb).reshape(-1)] if emb is not None else None,
            "codes": np.asarray(codes, dtype=int).tolist() if codes is not None else None
        }
        
        clean_filename = re.sub(r"[^a-zA-Z0-9_\-\.]", "_", voice_name) + ".json"
        local_json_path = os.path.join(LOCAL_EXPORT_DIR, clean_filename)
        with open(local_json_path, "w", encoding="utf-8") as f:
            json.dump(json_data, f, ensure_ascii=False, indent=2)
            
        drive_status = ""
        if os.path.exists(DRIVE_DIR):
            drive_json_path = os.path.join(DRIVE_DIR, clean_filename)
            shutil.copy(local_json_path, drive_json_path)
            drive_status = f"\n📁 Đã tự động lưu bản sao vào Google Drive: {drive_json_path}"
            
        SESSION_CLONED_VOICES[voice_name] = local_json_path
        
        audio_arr = tts.infer(test_text.strip(), voice=voice_name)
        preview_wav = os.path.join(tempfile.gettempdir(), f"preview_{int(time.time()*1000)}.wav")
        tts.save(audio_arr, preview_wav)
        
        msg = f"🎉 Clone thành công giọng: [{voice_name}]!\n" \
              f"- Vector đặc trưng Speaker Embedding: 192 chiều\n" \
              f"- File JSON sẵn sàng: {clean_filename} (~{os.path.getsize(local_json_path)/1024:.1f} KB){drive_status}\n\n" \
              f"👉 Bấm nút 'Tải file cấu hình giọng (.json)' bên dưới để lưu về máy tính!"
              
        return preview_wav, msg, local_json_path
    except Exception as e:
        import traceback
        err = traceback.format_exc()
        return None, f"❌ Lỗi khi clone giọng: {str(e)}\n\n{err}", None

def synthesize_text(voice_choice, text_input, speed):
    if not text_input or not text_input.strip():
        return None, "Văn bản rỗng!"
    try:
        clean_v = voice_choice
        if " · " in clean_v:
            clean_v = clean_v.split(" · ")[0].strip()
        audio_arr = tts.infer(text_input.strip(), voice=clean_v, speed=float(speed))
        out_wav = os.path.join(tempfile.gettempdir(), f"synth_{int(time.time()*1000)}.wav")
        tts.save(audio_arr, out_wav)
        return out_wav, f"✅ Đã đọc thành công bằng giọng [{clean_v}]!"
    except Exception as e:
        return None, f"❌ Lỗi: {str(e)}"

with gr.Blocks(title="VieNeu-TTS Voice Clone Studio 48kHz", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🦜 VieNeu-TTS v3 Turbo — Studio Nhân Bản Giọng Nói Tiếng Việt (48kHz)")
    gr.Markdown("> **Mô hình AI:** VieNeu-TTS v3 Turbo (Phạm Nguyễn Ngọc Bảo) • Chuẩn Studio 48kHz • Khử nhiễu tự động • Xuất file `.json` siêu nhẹ nạp vào Tool Review Video.")
    
    with gr.Tabs():
        with gr.TabItem("🎙️ 1. Nhân Bản Giọng Mới (Zero-Shot Clone)"):
            with gr.Row():
                with gr.Column(scale=1):
                    in_audio = gr.Audio(label="1. Tải lên file âm thanh mẫu (3–8 giây, WAV / MP3 / M4A):", type="filepath")
                    in_name = gr.Textbox(label="2. Đặt tên cho giọng clone:", placeholder="Ví dụ: Reviewer Nam Bựa, Cô Bé Dễ Thương...")
                    with gr.Row():
                        in_gender = gr.Dropdown(label="Giới tính:", choices=["Nam (Male)", "Nữ (Female)"], value="Nam (Male)")
                        in_region = gr.Dropdown(label="Vùng miền:", choices=["Bắc", "Trung", "Nam"], value="Bắc")
                    in_denoise = gr.Checkbox(label="Tự động khử tạp âm nền (Denoise)", value=True)
                    in_text = gr.Textbox(
                        label="3. Văn bản đọc thử nghiệm:",
                        lines=3,
                        value="[cười] Chào mừng mọi người đã quay trở lại với kênh review phim. Hôm nay chúng ta sẽ cùng phân tích một tình tiết vô cùng bất ngờ!"
                    )
                    btn_clone = gr.Button("🚀 Nhân Bản Giọng & Nghe Thử Âm Thanh 48kHz", variant="primary", size="lg")
                    
                with gr.Column(scale=1):
                    out_audio = gr.Audio(label="Âm thanh 48kHz đọc thử bằng giọng vừa clone:")
                    out_status = gr.Textbox(label="Trạng thái xử lý:", lines=6)
                    out_file = gr.File(label="Tải file cấu hình giọng (.json) về máy:")
                    
            btn_clone.click(
                fn=clone_and_preview,
                inputs=[in_audio, in_name, in_gender, in_region, in_text, in_denoise],
                outputs=[out_audio, out_status, out_file]
            )
            
        with gr.TabItem("⚡ 2. Đọc Thử Văn Bản Với Mọi Giọng"):
            with gr.Row():
                with gr.Column(scale=1):
                    preset_list = [f"{label} · {vid}" for label, vid in tts.list_preset_voices()]
                    synth_voice = gr.Dropdown(label="Chọn giọng đọc (Giọng có sẵn hoặc vừa clone):", choices=preset_list, value=preset_list[0] if preset_list else None)
                    synth_speed = gr.Slider(label="Tốc độ đọc:", minimum=0.6, maximum=1.5, value=1.0, step=0.05)
                    synth_text = gr.Textbox(
                        label="Văn bản cần đọc:",
                        lines=4,
                        value="VieNeu-TTS là mô hình chuyển văn bản thành giọng nói tiếng Việt chất lượng cao 48kHz. Bạn có thể sử dụng các tag cảm xúc như [cười], [thở dài], [hắng giọng] để giọng đọc sống động hơn."
                    )
                    btn_synth = gr.Button("🔊 Đọc Văn Bản", variant="primary")
                with gr.Column(scale=1):
                    synth_audio_out = gr.Audio(label="Kết quả âm thanh:")
                    synth_status = gr.Textbox(label="Thông báo:")
            btn_synth.click(
                fn=synthesize_text,
                inputs=[synth_voice, synth_text, synth_speed],
                outputs=[synth_audio_out, synth_status]
            )
            
        with gr.TabItem("📖 3. Hướng Dẫn Tích Hợp Vào Tool Review Video"):
            gr.Markdown("""
            ### 📌 Quy trình 3 bước đưa giọng vừa clone vào Tool trên máy tính:
            
            1. **Bước 1**: Sau khi bấm **'Nhân Bản Giọng'** ở Tab 1 và nghe thử ưng ý, bấm vào khung **'Tải file cấu hình giọng (.json)'** để lưu file `.json` về máy tính của bạn.
            2. **Bước 2**: Trên máy tính, copy file `.json` đó bỏ vào thư mục:
               👉 `Tool_Review_Video / vieneu_custom_voices /`
            3. **Bước 3**: Mở Tool Review Video:
               - Tại phần chọn giọng đọc, chọn động cơ **`VieNeu-TTS`**.
               - Giọng mới của bạn sẽ tự động xuất hiện với nhãn: **`[Clone] Tên_Giọng`**.
               - Bấm lồng tiếng video -> Tool sẽ đọc bằng **CPU nội bộ** máy tính hoàn toàn offline, chuẩn 48kHz siêu tự nhiên!
            """)

demo.launch(share=True, debug=False)
